## Gradio Day!

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
import gradio as gr

In [9]:
load_dotenv(override=True)
gemini_key = os.getenv("GEMINI_API_KEY")
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(base_url=gemini_url, api_key=gemini_key)

In [21]:
system_prompt = "You are a helpful assistant"

def message_gemini(prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    response = gemini.chat.completions.create(model="gemini-3.1-flash-lite-preview", messages=messages)
    return response.choices[0].message.content

In [ ]:
message_gemini("What is today's date?")

'Today is Wednesday, May 22, 2024.'

## User Interface time!

In [12]:
def shout(text):
    print(f"Shout has been called with input {text}")
    return text.upper()

In [13]:
shout("hello")

Shout has been called with input hello


'HELLO'

In [19]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True, auth=("ed","bananas"))

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Shout has been called with input Hi


In [ ]:
message_input = gr.Textbox(label="Your messages:", info="Enter a message to be shouted", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=shout,
    title="Shout",
    inputs=[message_input],
    outputs=[message_output],
    examples=["hello", "howdy"],
    flagging_mode="never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Shout has been called with input hi there how are you?

Shout has been called with input hello


In [3]:
ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(base_url=ollama_url, api_key="ollama")

In [4]:
system_prompt = "You are a helpful assistant"

def message_ollama(prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    response = ollama.chat.completions.create(model="llama3.2", messages=messages)
    return response.choices[0].message.content

In [25]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for gemini", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=message_ollama,
    title="OLLAMA",   
    inputs=[message_input],
    outputs=[message_output],
    examples=['hello', 'howdy'],
    flagging_mode="never" 
)
view.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [27]:
system_prompt = "You are a helpful assistant that responds in markdown without code blocks"

message_input = gr.Textbox(label="Your message:", info="Enter a message for ollama", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=message_ollama,
    title="ollama",
    inputs=[message_input],
    outputs=[message_output],
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
    ],
    flagging_mode='never'
)
view.launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


In [5]:
def stream_ollama(prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    stream = ollama.chat.completions.create(
        model="llama3.2",
        messages=messages,
        stream=True
    )

    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [6]:
def stream_gemini(prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    stream = gemini.chat.completions.create(
        model="gemini-3.1-flash-lite-preview",
        messages=messages,
        stream=True
    )

    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [32]:
system_prompt = "You are a helpful assistant that responds in markdown without code blocks"

message_input = gr.Textbox(label="Your message:", info="Enter a message for ollama", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_ollama,
    title="ollama",
    inputs=[message_input],
    outputs=[message_output],
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
    ],
    flagging_mode='never'
)
view.launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


### Now getting fancy

In [7]:
def stream_model(prompt, model):
    if model == "gemini":
        result = stream_gemini(prompt)
    elif model == "ollama":
        result = stream_ollama(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [14]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(["gemini", "ollama"], label="Select model", value="ollama")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_model,
    title="LLMs",
    inputs=[message_input, model_selector],
    outputs=[message_output],
    examples=[
        ["Explain the Transformer architecture to a layperson", "ollama"],
        ["Explain the Transformer architecture to an aspiring AI engineer", "gemini"]
    ],
    flagging_mode="never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [15]:
from scraper import fetch_website_contents

In [16]:
system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

In [17]:
def stream_brochure(company_name, url, model):
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    if model=="ollama":
        result = stream_ollama(prompt)
    elif model=="gemini":
        result = stream_gemini(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [18]:
name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
model_selector = gr.Dropdown(["ollama", "gemini"], label="Select model", value="ollama")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator", 
    inputs=[name_input, url_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Hugging Face", "https://huggingface.co", "ollama"],
            ["Edward Donner", "https://edwarddonner.com", "gemini"]
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
